# Qual é o prato? ConvNeXt-Tiny ajustado no Food-101

Este notebook ajusta o ConvNeXt-Tiny, pré-treinado no ImageNet, para reconhecer 101 pratos pela foto. O treino usa as 75.750 fotos de treino do Food-101 e a avaliação usa as 25.250 fotos de teste, que foram revisadas à mão pelos autores do dataset. As fotos de treino têm ruído de rótulo de propósito.

Para rodar, abra no Colab e escolha a GPU T4 em Ambiente de execução → Alterar o tipo de ambiente de execução.

In [ ]:
!pip install -q "transformers==5.22.1" "datasets==5.1.0" "accelerate==1.15.0"

## Configuração

O login pede um token do Hugging Face com permissão de escrita (Settings → Access Tokens). Ele vem no início para o modelo subir assim que o treino acabar.

In [ ]:
BASE = "facebook/convnext-tiny-224"
RUN = "convnext-tiny-food101-e3"
REPO = "juan1211/pratos-convnext-tiny-food101"
SEED = 2027
TAMANHO = 224

TREINO = dict(
    num_train_epochs=3,
    learning_rate=2e-4,
    per_device_train_batch_size=64,
    weight_decay=0.05,
)

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## Dados

No treino, cada foto passa por recorte aleatório e espelhamento. No teste, a foto é reduzida e cortada no centro, sempre igual. A normalização é a do pré-treino do ConvNeXt.

In [ ]:
import torch
from datasets import load_dataset
from torchvision import transforms
from transformers import AutoImageProcessor

ds = load_dataset("ethz/food101")
pratos = ds["train"].features["label"].names
processador = AutoImageProcessor.from_pretrained(BASE)
normalizar = transforms.Normalize(processador.image_mean, processador.image_std)

aumento = transforms.Compose(
    [transforms.RandomResizedCrop(TAMANHO), transforms.RandomHorizontalFlip(), transforms.ToTensor(), normalizar]
)
padrao = transforms.Compose(
    [transforms.Resize(int(TAMANHO * 1.14)), transforms.CenterCrop(TAMANHO), transforms.ToTensor(), normalizar]
)


def preparar(transformacao):
    def aplicar(lote):
        lote["pixel_values"] = [transformacao(foto.convert("RGB")) for foto in lote["image"]]
        return lote

    return aplicar


treino = ds["train"].with_transform(preparar(aumento))
teste = ds["validation"].with_transform(preparar(padrao))


def juntar(exemplos):
    return {
        "pixel_values": torch.stack([e["pixel_values"] for e in exemplos]),
        "labels": torch.tensor([e["label"] for e in exemplos]),
    }


print(len(treino), "fotos de treino,", len(teste), "de teste,", len(pratos), "pratos")

## Treino

A cabeça de classificação do ImageNet (1.000 classes) é trocada por uma de 101. O modelo inteiro é ajustado em fp16, com cosseno e aquecimento. O modelo sobe para o Hugging Face, como privado, logo depois do treino, para não se perder se o Colab desconectar durante a avaliação.

In [ ]:
import time

from transformers import AutoModelForImageClassification, Trainer, TrainingArguments

torch.manual_seed(SEED)
modelo = AutoModelForImageClassification.from_pretrained(
    BASE,
    num_labels=len(pratos),
    id2label=dict(enumerate(pratos)),
    label2id={nome: i for i, nome in enumerate(pratos)},
    ignore_mismatched_sizes=True,
)
args = TrainingArguments(
    output_dir=f"runs/{RUN}",
    seed=SEED,
    fp16=True,
    per_device_eval_batch_size=128,
    lr_scheduler_type="cosine",
    warmup_steps=200,
    logging_steps=100,
    save_strategy="no",
    dataloader_num_workers=2,
    remove_unused_columns=False,
    report_to="none",
    **TREINO,
)
trainer = Trainer(model=modelo, args=args, train_dataset=treino, data_collator=juntar)
inicio = time.perf_counter()
trainer.train()
minutos_treino = round((time.perf_counter() - inicio) / 60, 1)
trainer.model.push_to_hub(REPO, private=True)
processador.push_to_hub(REPO, private=True)
print(f"treino: {minutos_treino} min")

## Avaliação no teste

Acerto top-1 (o primeiro palpite é o prato certo) e top-5 (o prato certo está entre os 5 primeiros), os pratos com menor acerto e os pares que o modelo mais confunde.

In [ ]:
import json
from collections import Counter
from pathlib import Path

saida = trainer.predict(teste)
logits, rotulos = torch.tensor(saida.predictions), torch.tensor(saida.label_ids)
top5 = logits.topk(5, dim=1).indices
palpite = top5[:, 0]
acerto = palpite == rotulos

por_prato = {pratos[i]: round(acerto[rotulos == i].float().mean().item(), 3) for i in range(len(pratos))}
confusoes = Counter((pratos[r], pratos[p]) for r, p in zip(rotulos.tolist(), palpite.tolist(), strict=True) if r != p)

resultado = {
    "modelo": REPO,
    "fotos_teste": len(rotulos),
    "top1": round(acerto.float().mean().item(), 4),
    "top5": round((top5 == rotulos[:, None]).any(dim=1).float().mean().item(), 4),
    "piores_pratos": sorted(por_prato.items(), key=lambda item: item[1])[:10],
    "mais_confundidos": [{"real": r, "previsto": p, "fotos": n} for (r, p), n in confusoes.most_common(10)],
    "minutos_treino": minutos_treino,
    "gpu": torch.cuda.get_device_name(),
}
Path("reports").mkdir(exist_ok=True)
Path("reports/avaliacao_teste.json").write_text(json.dumps(resultado, indent=2, ensure_ascii=False), encoding="utf-8")
resultado

## Latência na CPU

O Space gratuito roda em CPU, então a latência que importa para a demonstração é a de uma foto por vez sem GPU.

In [ ]:
import statistics

cpu = trainer.model.float().to("cpu").eval()
amostra = torch.stack([teste[i]["pixel_values"] for i in range(30)])
tempos = []
with torch.inference_mode():
    for foto in amostra:
        inicio = time.perf_counter()
        cpu(pixel_values=foto[None])
        tempos.append(time.perf_counter() - inicio)
resultado["latencia_cpu_p50_ms"] = round(statistics.median(tempos) * 1000)
Path("reports/avaliacao_teste.json").write_text(json.dumps(resultado, indent=2, ensure_ascii=False), encoding="utf-8")
print(resultado["latencia_cpu_p50_ms"], "ms por foto na CPU")

## Salvar

O relatório vai para download, para entrar no repositório e no model card.

In [ ]:
from google.colab import files

files.download("reports/avaliacao_teste.json")